In [1]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)
print("Added to sys.path:", assignment_root)

from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/mayurakshi/Documents/Model To Markets/FRE-GY-9743-Assignments-1-fork
Fixed Income Library is loaded.


## Homework 1 --- 1-D Interpolation

Implement the four methods marked `## TODO` inside `Interpolator1DPCP`, in
`fixedincomelib/utilities/numerics.py`:

- `interpolate`
- `integrate`
- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

Then fill in `bump_reval_interpolator_integrand` further down in this notebook.

The interpolation convention is spelled out in the `Interpolator1DPCP`
docstring. Read it before writing.

To check yourself, run every cell in this notebook top to bottom. Each check
prints your value next to the expected one --- every `diff` should be around `0.0`.


### Test interpolation

In [2]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3, expected 3.0, diff = 0.0
f(1.0) = 3, expected 3.0, diff = 0.0
f(1.5) = 4, expected 4.0, diff = 0.0
f(3.0) = 4, expected 4.0, diff = 0.0
f(5.5) = 6, expected 6.0, diff = 0.0
f(6.5) = 6, expected 6.0, diff = 0.0
f(8.0) = 6, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [3]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [4]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1DIntegral(x_s, x_e, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)

### Interpolation sensitivity

In [5]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [6]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 7.571721027943568e-12
[1.5, 5.2]: max abs diff = 2.7955415760061442e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 1.1823431123048067e-10


# Answers: Open Questions

Prices $B$ and $K$ below are quoted per 100 face, so $B(T_s;T_s,T_m)-K$ is the buyer's payoff per 100 face. On a notional of 1,000,000 the total payoff is $\frac{1{,}000{,}000}{100}\big(B(T_s;T_s,T_m)-K\big)$. The strike is chosen so that the contract is worth zero at inception, i.e. $K=B(0;T_s,T_m)$, where $T_m$ is the maturity of the Treasury (10 years from today).

---

## Q1. Motivation, term sheet, and settlement

A Treasury has almost no default risk, but its price still moves with interest rates. For illustration, a modified duration of 8 implies an approximate 8% price decline for a one-percentage-point rise in yield (the true figure depends on the bond and ignores convexity). The client is long the forward, so it fixes today the price it will pay for the bond in a year. A pension fund or insurer that expects cash next year is a natural example, since it is protected against yields falling before it can invest. A par forward also involves no upfront purchase payment, although collateral or margin may be required before settlement under the CSA, so it can also be used to take a view that yields will fall without buying the bond.

The term sheet needs to pin down:

- the parties: the client as buyer (long) and Bank A as seller (short)
- the notional, 1,000,000 face value
- the exact underlying bond (CUSIP, coupon, maturity $T_m$), because different Treasuries have very different prices
- the settlement date $T_s$, one year after inception, when the bond has about 9 years left
- the strike $K$, per 100 face
- the price convention (clean or dirty) and the treatment of accrued interest
- physical or cash settlement
- day count, business-day conventions, and the CSA terms that determine the discount curve $df_{csa}$

On $T_s$ with physical settlement, Bank A delivers the bond (1,000,000 face) and the client pays $\frac{1{,}000{,}000}{100}K$, plus accrued interest if $K$ is quoted clean. With cash settlement only the difference $\frac{1{,}000{,}000}{100}\big(B(T_s;T_s,T_m)-K\big)$ is exchanged: Bank A pays the client if it is positive and receives it from the client if it is negative.

---

## Q2. Hedging with the spot bond and internal funding

Bank A is short the forward, so its payoff at $T_s$ is $K-B(T_s;T_s,T_m)$ per 100 face and it loses if bond prices rise. To remove this exposure, the desk buys the bond today at $B(0;0,T_m)$ and holds it. A gain or loss on the bond is then offset by the opposite loss or gain on the forward, and on $T_s$ the bank delivers a bond it already owns instead of buying one in the market. The hedge is static: it needs no rebalancing and no interest-rate model, which is why the forward can be priced from the cost of replicating it.

The purchase is funded by two internal trades. Let $h$ be the repo haircut. First, the trading desk repos the bond to the repo desk: it posts the bond as collateral, receives $(1-h)B(0;0,T_m)$ in cash, and agrees to repurchase the bond at $T_s$ at the term repo rate $R$. Second, the Treasury desk covers whatever the repo does not. If $h=0$, repo fully finances the purchase, so no additional unsecured funding is needed (Treasury may still intermediate cash and set internal funding charges). If $h>0$, Treasury funds the remaining $hB(0;0,T_m)$ at its internal funding rate $R_T$.

Q3 assumes $h=0$. Under simple interest, with the haircut funding outstanding until settlement and unchanged coupon treatment, a haircut changes the forward price by $hB(0;0,T_m)(R_T-R)T_s$, which is an increase if $R_T>R$.

| Time | Bond | Repo desk | Treasury | Client |
|---|---|---|---|---|
| $t=0$ | Buy, pay $B(0;0,T_m)$ | Lends $(1-h)B(0;0,T_m)$ | Lends $hB(0;0,T_m)$ | No purchase payment |
| $t_i\le T_s$ | Receive $c_i$ | Coupon credited against repo debt, as assumed in Q3 | None | None |
| $T_s$ | Bond returned | Repay principal + interest at $R$, net of coupons | Repay $hB(0;0,T_m)$ + interest at $R_T$ | Deliver bond, receive $K$ |

At the replication strike, terminal profit is zero under these assumptions, independently of the bond's settlement price. Residual risks are discussed in Q5.

---

## Q3. Forward price from the repo rate

We want $B(0;T_s,T_m)$ from the coupon schedule $\{(t_i,c_i)\}$, the term repo rate $R$ for $[0,T_s]$, and the spot price $B(0;0,T_m)$. All prices are per 100 face and dirty (clean plus accrued), since the dirty price is what is actually paid and financed. I assume no haircut ($h=0$), that coupons are credited against the financing balance and earn a financing credit at the same simple rate $R$ until settlement, and that a coupon paid exactly at $T_s$ is received by the bank before delivery (the bond is delivered ex-coupon).

Borrowing 1 unit of cash from the repo desk for $[0,T_s]$ means repaying $1+RT_s$ at $T_s$, so borrowing $B(0;0,T_m)$ means repaying $B(0;0,T_m)(1+RT_s)$ before coupons. The bond is held as collateral, so each coupon $c_i$ goes to the repo desk and reduces the debt. Received at $t_i$, it earns the financing credit until $T_s$, so at $T_s$ it is worth $c_i\big(1+R(T_s-t_i)\big)$. The bank's net cost of buying, financing, and holding the bond is the repayment minus these coupons:

$$B(0;T_s,T_m)=B(0;0,T_m)\,(1+RT_s)-\sum_{0<t_i\le T_s}c_i\,\big(1+R(T_s-t_i)\big),$$

and the strike is $K=B(0;T_s,T_m)$, so that $V(0)=0$.

This is a no-arbitrage price. If $K$ were higher, the bank could sell the forward and run the replication for a riskless gain. If $K$ were lower, an arbitrageur could buy the forward, short the spot bond, and invest the proceeds, assuming symmetric borrowing and lending access and no transaction costs.

As an example, take $B(0;0,T_m)=100$, $R=4\%$, $T_s=1$, and one coupon $c=2$ at $t_1=0.5$. The repayment is $100\times1.04=104.00$, the coupon is worth $2\times(1+0.04\times0.5)=2.04$ at $T_s$, and so $B(0;T_s,T_m)=104.00-2.04=101.96$.

A higher $R$ raises $K$ because funding costs are passed on. Larger coupons lower the forward price, and earlier coupons lower it when $R>0$ because they earn more financing credit. For a clean quote, subtract accrued interest at $T_s$. If $R$ is quoted with continuous compounding, replace $1+R\tau$ by $e^{R\tau}$.

---

## Q4. Mark-to-market for $t\in(0,T_s]$

Once the trade is on, $K$ stays fixed, but the fair forward price for delivery on $T_s$ moves with the spot bond price, the repo rate, and the coupons still to come. So at time $t$ we recompute the forward price with the Q3 argument:

$$B(t;T_s,T_m)=B(t;t,T_m)\,\big(1+R_t(T_s-t)\big)-\sum_{t<t_i\le T_s}c_i\,\big(1+R_t(T_s-t_i)\big),$$

using the same coupon-crediting, ex-coupon, and day-count assumptions as Q3. Here $B(t;t,T_m)$ is the current dirty price, $R_t$ is the current repo rate for $[t,T_s]$, and only coupons not yet paid are included. The gap to the strike is settled on $T_s$, so it is discounted with the CSA curve:

$$V(t)=df_{csa}(t,T_s)\,\big(B(t;T_s,T_m)-K\big).$$

This is the value per 100 face, as in equation (2), and the total MTM is $\frac{\text{Notional}}{100}V(t)$. If $V(t)>0$ the client is in the money and Bank A is out of the money. Bank A's forward position is worth $-V(t)$ and is hedged by its bond and repo positions, subject to the residual risks discussed in Q5. At $t=0$, $K=B(0;T_s,T_m)$ gives $V(0)=0$.

For an example, keep $K=101.96$ from Q3. At $t=0.6$ the coupon has been paid, the bond trades at $103$ (dirty), $R_t=4\%$, and $T_s-t=0.4$. Assume the CSA discount rate is also $4\%$ with simple interest. Then

- the forward price is $103\times(1+0.04\times0.4)=104.648$,
- the discount factor is $df_{csa}(0.6,1)=1/1.016\approx0.98425$,
- $V(0.6)=(104.648-101.96)/1.016\approx+2.6457$ per 100 face.

On 1,000,000 face, using the unrounded expression, the client's total MTM is

$$10{,}000\times\frac{104.648-101.96}{1.016}=\$26{,}456.69\quad\text{(rounded to cents)},$$

and Bank A's forward position is worth $-\$26{,}456.69$ before the hedge.

---

## Q5. Market risk and revenue

The static hedge removes the main exposures. The bond position offsets outright Treasury price (interest-rate) risk, and the term repo fixes the financing rate over $[0,T_s]$, so, on contractual terms, the desk's payoff at $T_s$ is known at $t=0$ and does not depend on $B(T_s;T_s,T_m)$. That does not make the desk risk-free. Some residual risks are market-type risks: the repo curve can move relative to the CSA discount curve (basis), which affects the mark-to-market and any early unwind, and repo specialness matters if the locked term repo has to be replaced or unwound early. Haircut and margin funding can create extra liquidity needs during the trade, and there is also counterparty, liquidity, and settlement risk. Holding the term repo to maturity fixes its contractual financing rate, but additional collateral funding costs, counterparty default, or settlement failures can still affect realised profit.

At the exact replication price $K=B(0;T_s,T_m)$, the theoretical profit is zero: the desk delivers the bond for exactly what it costs to buy, fund, and hold it. To earn revenue it needs a spread, and the repo rate is the natural place for it. The Q3 forward price increases with $R$, so the desk quotes the client using $R_{\text{quote}}=R+s$ with $s>0$ in the Q3 formula, while still funding itself at the true rate $R$. Revenue at $T_s$ is

$$\frac{\text{Notional}}{100}\,\big(K_{\text{quote}}-K_{\text{fair}}\big).$$

With the Q3 inputs, $K_{\text{fair}}=101.96$ at $R=4.0\%$, while $R_{\text{quote}}=4.3\%$ gives $K_{\text{quote}}=102.257$. The spread is $0.297$ per 100 face, about $\$2{,}970$ on 1,000,000 face. The markup is fixed in the contract, but realised profit is not risk-free. It can be affected by counterparty default, early unwind or replacement of the repo, additional haircut and margin funding costs, balance-sheet costs, and transaction and settlement costs, which is what the spread compensates for.

To convince the client, the strongest argument is a funding advantage. The bank may fund the bond more cheaply than the client, so it can quote an implied repo rate above its own funding cost but below the client's alternative funding rate. The client gets competitive financing and the bank keeps a spread. In the example, if the client's own funding cost is $4.6\%$, self-financing the same position implies $K=102.554$, so the quote of $102.257$ is $0.297$ cheaper for the client and $0.297$ above fair value for the bank. The client also fixes its purchase price with no upfront bond purchase payment (collateral may be required), and the bank provides execution, hedging, and balance sheet while bearing the residual risks above.

---

## \*Q5. Relation to risk-neutral pricing

The replication approach in Q2–Q3 agrees with risk-neutral pricing under the $T_s$-forward measure when funding, discounting, and coupon accumulation assumptions are consistent.

Let $P(t,T)$ be the discount factor on the funding curve, $B(t)=B(t;t,T_m)$ the dirty bond price, and $c_i$ the deterministic coupons at $t_i$, with the ex-coupon convention for $0<t_i\le T_s$. Consider the portfolio that is long the bond and short zero-coupon bonds paying $c_i$ at $t_i$:

$$Y(t)=B(t)-\sum_{t<t_i\le T_s}c_i\,P(t,t_i).$$

The coupons cancel against the short zeros, so $Y$ is a traded asset with no payments before $T_s$, and $Y(T_s)=B(T_s)$. Taking $P(t,T_s)$ as the numeraire, $Y/P(\cdot,T_s)$ is a martingale under $\mathbb{Q}^{T_s}$, so

$$\frac{Y(0)}{P(0,T_s)}=\mathbb{E}^{T_s}[B(T_s)].$$

The forward has value $V(0)=P(0,T_s)\,\mathbb{E}^{T_s}[B(T_s)-K]$, and setting it to zero gives

$$K=\mathbb{E}^{T_s}[B(T_s)]=\frac{B(0;0,T_m)-\sum_{0<t_i\le T_s}c_i\,P(0,t_i)}{P(0,T_s)}.$$

The same argument at $t>0$ gives $V(t)=P(t,T_s)\big(\mathbb{E}^{T_s}[B(T_s)\mid\mathcal{F}_t]-K\big)$, which is the Q4 formula with $df_{csa}=P$.

To compare with Q3, suppose the repo curve is the discount curve. Define the simple repo rate $R$ by $1/P(0,T_s)=1+RT_s$, and the coupon accumulation factors by $P(0,t_i)/P(0,T_s)=1+F(t_i,T_s)(T_s-t_i)$, where $F$ is the simple forward repo rate. Then

$$K=B(0;0,T_m)(1+RT_s)-\sum_{0<t_i\le T_s}c_i\big(1+F(t_i,T_s)(T_s-t_i)\big).$$

This is the curve-consistent version of the Q3 replication formula. Q3 assumes coupons receive a simple financing credit at the same rate $R$; the two expressions coincide when the curve-implied coupon accumulation factors equal those assumed in Q3. Under a flat continuously compounded rate $r$, both approaches give exactly

$$K=B(0;0,T_m)\,e^{rT_s}-\sum_{0<t_i\le T_s}c_i\,e^{r(T_s-t_i)}.$$

In both, spot grown at the funding rate is $B(0;0,T_m)/P(0,T_s)$, and a coupon credited to the repo desk and grown to $T_s$ is $c_i\,P(0,t_i)/P(0,T_s)$.

As a numerical check, take a flat $4\%$ continuously compounded rate, $B=100$, one coupon $c=2$ at $t=0.5$, and $T_s=1$. Replication gives $100e^{0.04}-2e^{0.02}=104.0811-2.0404=102.0407$, and the risk-neutral formula gives $(100-2e^{-0.02})/e^{-0.04}=98.0396/0.960789=102.0407$. Q3's $101.96$ differs only because it accumulates at $4\%$ simple interest, so different accumulation assumptions give different prices, as expected.

The assumptions are: deterministic coupons and no default risk (Treasury); a repo curve equal to the discount (CSA) curve, with no repo spread, specialness, or basis; no haircuts, margin, or transaction costs; the ex-coupon settlement convention; and no arbitrage. Interest rates themselves may be stochastic.

If repo funding differs from CSA discounting, the single-curve derivation above no longer applies directly. A consistent risk-neutral valuation must incorporate repo financing and collateral terms and should agree with replication under the same assumptions. The dealer's contractual markup in Q5 is separate from this funding basis.